In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

### 1. Create Data Dummy

In [2]:
# Membuat data dummy dengan 12 fitur dan 1 target
X, y = make_classification(n_samples=3480, n_features=12, n_informative=8, n_redundant=4, random_state=42)

In [3]:
# Menjadikan data sebagai DataFrame untuk kemudahan visualisasi
df = pd.DataFrame(X, columns=[f'Feature_{i+1}' for i in range(X.shape[1])])
df['Target'] = y

In [4]:
df.sample(3)

,Feature_1,Feature_2,Feature_3,Feature_4,Feature_5,Feature_6,Feature_7,Feature_8,Feature_9,Feature_10,Feature_11,Feature_12,Target
1752,1.278388,2.685788,4.465860,-1.440487,0.355574,0.519826,3.341286,-0.736044,0.582634,0.297913,4.400724,-3.384007,1
630,-0.197556,3.837071,-3.882634,-0.946024,1.671839,1.746403,0.732410,2.192457,-2.754294,1.186218,2.383489,1.375276,0
1772,-2.127652,1.622412,-3.193011,-2.405618,0.200191,-1.069016,-1.563206,-1.257253,-1.552000,1.494195,1.515269,0.261223,0


In [5]:
df.shape

(3480, 13)

### 2. Evaluasi dan Seleksi Fitur

#### 2.1 Mutual Information

Mutual Information (MI) mengukur ketergantungan antara dua variabel. Ini memberikan pemahaman tentang seberapa besar informasi yang dibagikan oleh dua variabel (dalam hal ini, fitur dan target).

In [6]:
from sklearn.feature_selection import mutual_info_classif

# Menghitung Mutual Information antara fitur dan target
mi = mutual_info_classif(df.drop(columns='Target'), df['Target'])

# Menampilkan hasil mutual information
mi_df = pd.DataFrame({'Feature': df.drop(columns='Target').columns, 'Mutual Information': mi})
mi_df = mi_df.sort_values(by='Mutual Information', ascending=False)

In [7]:
print("Mutual Information Ranking:")
print(mi_df)

Mutual Information Ranking:
       Feature  Mutual Information
0    Feature_1            0.121676
4    Feature_5            0.079360
10  Feature_11            0.052454
8    Feature_9            0.041628
11  Feature_12            0.040050
7    Feature_8            0.038974
2    Feature_3            0.012375
3    Feature_4            0.011766
9   Feature_10            0.011016
1    Feature_2            0.010661
5    Feature_6            0.008885
6    Feature_7            0.000000


#### 2.2 Recursive Feature Elimination (RFE)

RFE adalah teknik yang menghapus fitur satu per satu, berdasarkan kinerja model. Fitur yang paling tidak berpengaruh dihapus terlebih dahulu.

In [10]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Menggunakan model Logistic Regression untuk RFE
model = LogisticRegression(max_iter=10000)

# Melakukan RFE dengan 5 fitur teratas
# Melakukan RFE dengan menyebutkan argumen secara eksplisit
rfe = RFE(estimator=model, n_features_to_select=5)
rfe.fit(df.drop(columns='Target'), df['Target'])

,estimator,LogisticRegre...ax_iter=10000)
,n_features_to_select,5
,step,1
,verbose,0
,importance_getter,'auto'
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1


In [11]:
# Menampilkan fitur yang dipilih oleh RFE
rfe_selected_features = pd.DataFrame({'Feature': df.drop(columns='Target').columns, 'Selected': rfe.support_})
rfe_selected_features = rfe_selected_features[rfe_selected_features['Selected'] == True]

In [12]:
print("Fitur yang dipilih oleh RFE:")
print(rfe_selected_features)

Fitur yang dipilih oleh RFE:
     Feature  Selected
1  Feature_2      True
2  Feature_3      True
4  Feature_5      True
5  Feature_6      True
8  Feature_9      True


#### 2.3 Model-Based Feature Selection (Menggunakan Random Forest)

Model-Based Feature Selection menggunakan model seperti Random Forest untuk menentukan pentingnya fitur berdasarkan kontribusinya dalam memprediksi target.

In [13]:
from sklearn.ensemble import RandomForestClassifier

# Menggunakan Random Forest untuk seleksi fitur
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(df.drop(columns='Target'), df['Target'])

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [14]:
# Menampilkan pentingnya fitur berdasarkan model Random Forest
rf_feature_importance = pd.DataFrame({'Feature': df.drop(columns='Target').columns, 'Importance': rf.feature_importances_})
rf_feature_importance = rf_feature_importance.sort_values(by='Importance', ascending=False)

In [15]:
print("Pentingnya Fitur berdasarkan Random Forest:")
print(rf_feature_importance)

Pentingnya Fitur berdasarkan Random Forest:
       Feature  Importance
0    Feature_1    0.175875
7    Feature_8    0.134446
4    Feature_5    0.111988
10  Feature_11    0.107134
9   Feature_10    0.081209
8    Feature_9    0.070291
1    Feature_2    0.061420
5    Feature_6    0.060275
11  Feature_12    0.051969
6    Feature_7    0.050305
3    Feature_4    0.048768
2    Feature_3    0.046320


### 3. Menggabungkan Hasil Seleksi Fitur

In [16]:
# Gabungkan hasil seleksi dengan menggunakan fitur yang dipilih oleh ketiga metode
selected_features = {
    'Mutual Information': mi_df['Feature'].head(5).values,  # Menampilkan 5 fitur teratas dari MI
    'RFE': rfe_selected_features['Feature'].values,
    'Random Forest': rf_feature_importance['Feature'].head(5).values  # Menampilkan 5 fitur teratas dari RF
}

In [17]:
# Menampilkan fitur yang terpilih
print("Fitur Terpilih dari Ketiga Teknik Seleksi:")
for method, features in selected_features.items():
    print(f"\n{method}:")
    print(features)

Fitur Terpilih dari Ketiga Teknik Seleksi:

Mutual Information:
['Feature_1' 'Feature_5' 'Feature_11' 'Feature_9' 'Feature_12']

RFE:
['Feature_2' 'Feature_3' 'Feature_5' 'Feature_6' 'Feature_9']

Random Forest:
['Feature_1' 'Feature_8' 'Feature_5' 'Feature_11' 'Feature_10']
